# 屋頂＋植物：從植物條件到 STL

老師的驗收是：**設定屋頂條件＋分支條件 → 產生符合兩者的 SDF／STL**。這本 notebook 從頭跑一次：

| 格 | 做什麼 | 看什麼 |
|---|---|---|
| 1 | 讀屋頂條件 | 18 × 12 m、板厚、花台與排水點 |
| 2 | 讀植物條件 | 每種植物的三個數字，**全是假設值** |
| 3 | 換算成花台開口 | 每個花台的開口表 |
| 4 | 跑 Step 3–8 | 原版與植物版的平面對照 |
| 5 | 檢查 | 土深、壁厚、梁、採光口、排水點、積水、花台排水孔 |
| 5b | 板下排水管（v2） | 每個排水孔接到最近落水點的管路：長度、坡度、落差、有沒有穿過別的孔或採光口 |
| 6 | 剖面與 STL | 老師的驗收物 |
| 7 | 換一個花台的植物再跑 | 改條件，形體跟著變；只出表格和平面，不出 STL 與剖面 |
| 8 | 載重情境比較（v2） | 三種土壤情境、五種板密度並列；**只比較，不選、不判定結構安全** |
| 9 | 連通花台（v2，可選） | 兩個花台畫成一個；預設不跑 |

第二版（v2）起，每個花台底部都有一個排水孔，從花台底穿出板底（第 4、5 格）。

**2026-11-09 起 `group3_upright` 的 `depth` 是 400 mm（Wei 依《建築基地綠化設計技術規範》核准）。它在 P1 放不下（最多 366 mm），第 4 格會先顯示放不下的訊息，再改用暫代配置 `planter_plants_fallback.json`（P1 暫種 `control`）把後面跑完。暫代不是設計；要怎麼讓它放得下由 Wei 決定，選項見 `projects/roof-plants/docs/roof-plants.md` v2 第 5 項（選項表）。**

全部跑完約 1 分鐘（不含 53 塊的完整 build）。**改第 1～3 格的內容，要從改過的那一格一路重跑到底**；第 7 格的兩個值改了只重跑第 7 格。植物程式都在 `roof_plants/plant*.py`，這裡只呼叫、不定義函式。怎麼和其他分支合併，見 `projects/roof-plants/docs/roof-plants.md`。

## 1. 屋頂條件

`brief` 是所有數字（滑桿），`drawing` 是手繪的開口、花台和排水點。和 `build.py` 一樣，資料夾裡有 `brief.json` 就讀它，沒有就用預設值。

In [ ]:
import json
import os

import matplotlib.pyplot as plt

import roof_plants as roof
from compas_vol.meshing import export_stl
from roof_plants.plant_build import build_pieces
from roof_plants.plant_checks import DRAIN_GAP, table, water_check
from roof_plants.plant_drains import (all_ok_drained, drain_rows, drain_table, drained_planter_rows,
                                    plant_steps_drained)
from roof_plants.plant_figures import plan_comparison, section_comparison
from roof_plants.plant_loads import compare, plant_weights, scenario_table, slab_table
from roof_plants.plant_pipes import (pipe_overlaps, pipe_rows, pipe_table, planter_pipes, plot_plan,
                                   plot_profiles)
from roof_plants.plants import PlanterFitError, assign, load_plants, opening_spec, reassign

%matplotlib inline

brief_source = "brief.json" if os.path.exists("brief.json") else "Brief() defaults"
brief = roof.Brief.load("brief.json") if os.path.exists("brief.json") else roof.Brief()
drawing = roof.Drawing.load("drawing.json")
OUT = "out"
RUN_ALL = False          # True: build all 53 pieces at the end of cell 6, a few minutes
RUN_CONNECTED = False    # True: run cell 9, the connected planters, about 30 s
os.makedirs(OUT, exist_ok=True)

print("brief from:", brief_source)
print("roof {:.0f} x {:.0f} mm, thickness {:.0f}, planter_wall {:.0f}".format(
    brief.length, brief.width, brief.thickness, brief.planter_wall))
print("drawing: {} openings, {} planters, {} drains".format(
    len(drawing.openings), len(drawing.planters), len(drawing.drains)))

## 2. 植物條件

> **這些數字都是假設值，沒有文獻來源。** 這次要證明的是流程：植物條件改了，形體跟著變；不是證明這些數字對植物合適。之後要換成由根系類型、生長方向和根深換算（`opening_spec` 裡有註記）。

- `offset`：花台底的輪廓比手繪線外擴（＋）或內縮（−）多少
- `rim`：從底到頂再往外張多少，像喇叭口
- `depth`：花台底有多深
- `depth_datum`：`depth` 從哪裡量。`nominal` 從板頂 z = 600 量（原版做法，對照組用）；`local_min` 從**花台範圍內最低的實際板面**量，所以 `depth` 就是最淺處的土深

In [ ]:
plants = load_plants("plants.json")
with open("planter_plants.json", encoding="utf-8") as f:
    planter_plants = json.load(f)

for plant in plants.values():
    print("{id:<16} {name_zh}: offset {offset:+g}, rim {rim:g}, depth {depth:g} ({depth_datum})".format(**plant))
    print("    source: {source}".format(**plant))
print()
print("planters:", [row["plant"] for row in planter_plants["planters"]])

## 3. 換算成花台開口

`opening_spec` 目前只檢查再傳下去；之後根系換算也只改這個函式，後面都不用動。`top` 是 `offset + rim`，也就是手繪線在板頂外擴多少。

In [ ]:
specs = [opening_spec(p) for p in assign(plants, planter_plants, len(drawing.planters))]

print("P   plant             offset   rim  depth  top    datum")
for n, spec in enumerate(specs):
    print("P{}  {id:<16} {offset:>6.0f} {rim:>5.0f} {depth:>6.0f} {top:>5.0f}  {depth_datum}".format(n, **spec))

## 4. 跑 Step 3–8，看平面

同一個 `design_roof` 跑兩次：不加參數是原版；加 `steps=plant_steps_drained(...)` 就讓 Step 4 看到花台的新輪廓、Step 7 依植物挖，並在每個花台底部開一個排水孔（v2）。

花台放不下時，`design_roof` 在 Step 4 之後就會停，並告訴你 `depth` 最多多少。這格先試 `planter_plants.json`（設計意圖）；放不下就印出原因，改用 `planter_plants_fallback.json`（暫代，P1 種 `control`）跑完，之後每一格用的都是暫代配置，`PLANTING` 會寫明。

平面圖的實線是花台底，虛線是頂面開口。

In [ ]:
original = roof.design_roof(brief, drawing)
PLANTING = "planter_plants.json"
try:
    planted = roof.design_roof(brief, drawing, steps=plant_steps_drained(brief, specs))
except PlanterFitError as error:
    print("THE INTENDED PLANTING DOES NOT FIT:", error)
    PLANTING = "planter_plants_fallback.json"
    with open(PLANTING, encoding="utf-8") as f:
        planter_plants = json.load(f)
    specs = [opening_spec(p) for p in assign(plants, planter_plants, len(drawing.planters))]
    print("STAND-IN, not the design: from here on", PLANTING, "->",
          [row["plant"] for row in planter_plants["planters"]])
    planted = roof.design_roof(brief, drawing, steps=plant_steps_drained(brief, specs))
SUFFIX = "_standin" if PLANTING == "planter_plants_fallback.json" else ""   # on every file made from it
print("planting:", PLANTING)

fig = plan_comparison(brief, drawing, original, planted, specs, os.path.join(OUT, "plan_compare.png"))

## 5. 檢查

每個花台一列，單位 mm、m²。

| 欄位 | 意思 | 規則 |
|---|---|---|
| `floor_z` | 花台底的高度 | |
| `wall_margin` | 底下留完壁厚後還剩多少 | ≥ 0 |
| `kept` | 花台底下實際保留的厚度 | 約等於壁厚 150 |
| `wall_normal` | 花台壁**垂直於壁面**的最薄厚度（公式算的） | 只回報，不影響 `ok` |
| `soil_min`／`soil_max` | 花台底上的實際土深（板面有洩水坡和水道，所以不一樣） | `soil_min` > 0 |
| `beam_gap` | 頂面開口到梁的距離 | > 0 |
| `light_gap` | 頂面開口到採光口實際形狀的距離 | > 壁厚 150，要放得下壁 |
| `drain_gap` | 頂面開口到最近排水點的距離 | ≥ `DRAIN_GAP`，**這是暫定值**；不論怎麼設，都不能小於壁厚＋出水口半徑 230 mm |
| `area_m2` | 板面下 10 mm 實測的開口面積 | 只回報 |

第二張表是排水孔（v2）：

| 欄位 | 意思 | 規則 |
|---|---|---|
| `bores` | 這個花台有幾個排水孔（花台底分成幾塊就幾個） | ≥ 1 |
| `radius` | 孔的半徑，預設等於 `brief.outlet` 80 mm，**是假設值** | |
| `open` | 沿孔軸和半徑一半的圓上，從花台底上 20 mm 到板底上 5 mm 取樣，設計、無泡沫版、`keep` 三者都是空的比例 | 100% |
| `beam_gap` | 孔邊到最近的梁的平面距離 | > 0，孔不吃梁的 500 mm |

排水孔是花台凹槽的一部分，所以 `keep` 的花台壁會沿著孔包一圈 150 mm 的套管一路到板底，不會把孔填回去。`kept` 欄量的是孔以外的花台底下有沒有完整的壁；孔本身由第二張表檢查。

**壁厚 150 的意思**：`planter_wall` 是水平量的，所以喇叭口越斜，垂直於壁面的厚度就越薄。`wall_normal` 把這件事列出來；開口外張最多的 P2 大約只有 73 mm。要不要改 `rim` 或壁厚還沒決定。

**積水的定義**：`stranded` 是水完全流不到任何出水口的格數，`reverse_falls` 是水道中比下游還低的格數。小窪地會先被填滿再流出去，**不算**在 `stranded` 裡，所以「積水 0」不代表板面完全沒有水窪。

對照組 `soil_min` 比 350 淺很多，是原版量法本來就有的現象；留著它，才看得出新量法差在哪。

In [ ]:
print("planting:", PLANTING, "(STAND-IN, not the design)" if SUFFIX else "")
rows = drained_planter_rows(brief, drawing, specs, planted)
water = water_check(planted.incision)
drains = drain_rows(brief, drawing, specs, planted)

print(table(rows))
print()
print(drain_table(drains))
print()
print("drain gap threshold (provisional):", DRAIN_GAP, "mm")
print("water:", water)
print("all checks pass:", all_ok_drained(rows, water, drains))

## 5b. 板下排水管（v2）

每個花台的排水孔從板底接一條管，吊在板下走到**最近的落水點**，在那裡接上落水點自己的豎管（豎管沒有建模）。管頂貼齊板底，沿梁格的兩個方向走（先 X 後 Y，不行就先 Y 後 X，都不行才走直線），一路以 `slope` 下降。

管線**沒有挖進屋頂的 SDF**，只是資料，加上這裡的平面和縱斷面圖。

| 欄位 | 意思 | 規則 |
|---|---|---|
| `length` | 管的水平長度 | 只回報 |
| `slope` | 坡度，預設 1 %，**PROVISIONAL（2026-11-09），沒有來源** | |
| `drop` | 坡度造成的落差 = 長度 × 坡度 | 只回報 |
| `below_soffit` | 管底在板底下多深（落差 + 管徑） | **板下可用空間沒有定義，只回報、不判定** |
| `clear` | 平面上有沒有穿過別的排水孔、別的落水點、採光口（各外擴管半徑） | 要 True |

FLL 要求植栽區內的落水頭要設檢查井，本工具沒有建模。

In [ ]:
pipes = planter_pipes(brief, drawing, specs, planted)      # slope=0.01 by default, PROVISIONAL
print("planting:", PLANTING, "(STAND-IN, not the design)" if SUFFIX else "")
print(pipe_table(pipes))
print("pipes closer in plan than their radii (reported, not judged):", pipe_overlaps(pipes))

fig, (ax_plan, ax_profile) = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={"width_ratios": [1.6, 1]})
roof.preview.plan(brief, drawing, planted.beams, planted.incision, ax=ax_plan, title="pipes under the slab (plan)")
plot_plan(ax_plan, pipes, drawing)
plot_profiles(ax_profile, pipes)
ax_profile.set_title("fall along each pipe")
fig.savefig(os.path.join(OUT, "pipes" + SUFFIX + ".png"), dpi=110)
plt.show()
with open(os.path.join(OUT, "pipes" + SUFFIX + ".json"), "w", encoding="utf-8") as f:
    json.dump(pipe_rows(pipes), f, indent=1)

## 6. 剖面與 STL

灰色是實體，紅線是花台，橘線是一定要留下的部分，藍色虛線是花台底，綠線是實際板面。

這裡輸出一個整屋的粗略 STL。用暫代配置時，STL 與剖面圖的檔名都加 `_standin`。要 53 塊都做（含風化、接頭、封閉檢查），把第 1 格的 `RUN_ALL` 改成 `True`，或在終端機跑 `uv run python -m roof_plants.plant_build --out out`。

In [ ]:
print("planting:", PLANTING, "(STAND-IN, not the design)" if SUFFIX else "")
fig = section_comparison(brief, drawing, original, planted, specs,
                         path=os.path.join(OUT, "section_compare" + SUFFIX + ".png"))

bounds = (-100, -100, -100, brief.length + 100, brief.width + 100, brief.thickness + 100)
preview_stl = os.path.join(OUT, "roof_plant_preview" + SUFFIX + ".stl")
export_stl(planted.design.roof, preview_stl, bounds, (183, 123, 11))
print("wrote", preview_stl, "{:.1f} MB".format(os.path.getsize(preview_stl) / 1e6))

if RUN_ALL:
    result = build_pieces(brief, planted, OUT, merged="roof_plant" + SUFFIX + ".stl")
    print("{watertight} of {count} pieces watertight; least beam depth {least_beam_depth} mm".format(**result))

## 7. 換一個花台的植物，再跑一次

現場示範：改下面兩個值，只重跑這一格。這格只出表格和平面，**不出 STL 與剖面**，要出的話，把換好的配置寫到**另一個檔**（例如 `planter_plants_try.json`），再用 `uv run python -m roof_plants.plant_build --planters projects/roof-plants/planter_plants_try.json` 指定。**不要覆寫 `planter_plants.json`**：它記的是設計意圖，現在是暫代模式，覆寫會讓 P1 種 `group3_upright` 這個意圖消失。

改第 1～3 格的內容（brief、drawing、植物資料）時，要從改過的那一格一路重跑到底，否則前後格會用到新舊不同的資料，而且不會報錯。

- `PLANTER`：第幾個花台（0–3）
- `PLANT`：`control`、`group1_creeping`、`group3_upright`

已試過的組合，以暫代配置（P1 `control`）為底，2026-11-09 實測，`group3_upright` 的 `depth` 為 400：

| 換法 | 結果 |
|---|---|
| P2 → `control`（預設） | 全部通過，開口從 4.96 縮到 1.99 m² |
| P0 → `group1_creeping` | 跑得完，但 `drain_gap` 420 不到暫定的 500，`ok` 變 `False` |
| P3 → `group1_creeping` | 跑得完，但頂面開口碰到梁（`beam_gap` −194），`ok` 變 `False` |
| P0／P1／P2／P3 → `group3_upright` | 都放不下，`depth` 最多分別 306／366／332／391 mm |

放不下不是壞掉，是檢查在工作。v1（`depth` 360）時 P3 → `group3_upright` 放得下，現在不行。

**會不會連累別的花台？** 機制上會：換植物可能改變水路，而 `local_min` 的花台底跟著板面走。v1（`group3_upright` 的 `depth` 360）時，在**預設 brief、目前的 drawing、只有這三種植物**的範圍內，81 種配置全掃過：36 種放得下（不代表其他檢查都通過），45 種放不下，放不下的都是自己種 `group3_upright` 的 P0 或 P2，沒換植物的花台一次都沒有報錯。這個掃描沒有用 `depth` 400 重做。這個結果不能外推：改 `fall`、`relief`、`relief_seed`、水道參數、排水點、花台或採光口位置，或加入新 `offset` 的植物，都要重跑。

In [ ]:
PLANTER = 2
PLANT = "control"

swapped = [opening_spec(p) for p in assign(plants, reassign(planter_plants, PLANTER, PLANT), len(drawing.planters))]
try:
    again = roof.design_roof(brief, drawing, steps=plant_steps_drained(brief, swapped))
except PlanterFitError as error:
    print("does not fit, stopped before the heavy steps:", error)
else:
    rows_again = drained_planter_rows(brief, drawing, swapped, again)
    water_again = water_check(again.incision)
    drains_again = drain_rows(brief, drawing, swapped, again)
    print(table(rows_again))
    print("water:", water_again)
    print("all checks pass:", all_ok_drained(rows_again, water_again, drains_again))
    print("opening of P{}: {:.2f} -> {:.2f} m2".format(
        PLANTER, rows[PLANTER]["deck_area"], rows_again[PLANTER]["deck_area"]))
    fig = plan_comparison(brief, drawing, planted, again, swapped, before=specs)
    plt.show()

## 8. 載重情境比較（v2）

**只比較，不選、不判定。** 這格把質量加總成 kN 與 kN/m²，並列幾種情境，不和任何容許值比較；結構安全不在這個工具的範圍。

- 花台：凹槽內「花台底以上、板面以下」的體積（SDF 取樣積分，25 mm 一格）× 飽和土壤密度，加植物（依 `plants.csv` 的 `mature_weight_kg_m2`，FLL 2018 表 26 p.117）與積水。
- 板：`design.roof` 的實體體積（50 mm 一格）× 板密度。風化與接頭前的設計，所以偏多。
- 全屋：53 塊加總，kN/m² 用 18 × 12 m。

| 情境 | 飽和土壤 | 積水 | 出處 |
|---|---|---|---|
| `light_medium` 輕介質 | 1,400 kg/m³ | 0 | ZinCo Info Sheet 2026-01 p.1（飽和 1,300–1,500） |
| `mineral_soil` 礦物土 | 1,800 kg/m³ | 0 | FLL 2018 表 24 p.115（密集型／礦物混合 1,600–1,900） |
| `drain_blocked` 排水孔堵塞 | 1,800 kg/m³ | 50 mm | 積水 50 mm 是 **Wei 未定，PROVISIONAL** |

「飽和」照 FLL 的最大含水量：先飽和、再自由排水（FLL 為滴水 2 小時）後保留的水分。

**板的材料沒有任何地方指定。** 預設 2400 kg/m³（PROVISIONAL，≈ EN 1991-1-1 表 A.1 一般混凝土 24 kN/m³ ×100 簡化）；第二張表並列五種密度，只列數字、不選材料。附錄 A 的值是乾燥狀態，板吸水後的濕重沒有計入。

In [ ]:
compared = compare(brief, drawing, specs, planted, plant_weights=plant_weights(plants))

print("planting:", PLANTING, "(STAND-IN, not the design)" if SUFFIX else "")
print("plants:", compared["plant_weights_note"], compared["plant_weights"])
print("saturated loads by scenario (slab 2400 kg/m3, PROVISIONAL):")
print(scenario_table(compared))
print()
print("slab density (material unspecified), and the roof's saturated kN/m2 under each scenario:")
print(slab_table(compared))
for density, material, source in [(r["density"], r["material"], r["source"]) for r in compared["slab"]]:
    print("  {:>6.0f}  {}: {}".format(density, material, source))
with open(os.path.join(OUT, "loads" + SUFFIX + ".json"), "w", encoding="utf-8") as f:
    json.dump(compared, f, indent=1, ensure_ascii=False)

## 9. 連通花台（v2，可選）

把 `drawing.json` 的 P1 和 P3 用 **800 mm** 寬的頸部畫成一個啞鈴形花台（`drawing_connected.json`，由 `uv run python projects/roof-plants/scripts/make_connected_drawing.py` 產生），其他不變；程式邏輯完全沒改。花台變成 3 個：P0、啞鈴、P2。

預設不跑，把第 1 格的 `RUN_CONNECTED` 改成 `True`。

| 啞鈴種什麼 | 結果（2026-11-09 實測，有測試鎖住） |
|---|---|
| `group3_upright`，`depth` 400（`plants.json` 現值） | **放不下**：啞鈴含 P1 原本最低的板面，`depth` 最多 366 mm，和 P1 單獨時一樣 |
| `group3_upright`，`depth` 360（v1 值，只作對照） | 全部檢查通過 |
| `control` | 全部檢查通過 |

頸部 500 mm 時，Step 4 把花台底輪廓內縮 300 mm 會切成兩圈，`offset_outline` 只接受一圈而停下；這個案例留在測試裡（`projects/roof-plants/tests/test_plant_connected.py`），不再放成檔案。

In [ ]:
if RUN_CONNECTED:
    connected = roof.Drawing.load("drawing_connected.json")
    with open("planter_plants_connected.json", encoding="utf-8") as f:
        connected_plants = json.load(f)
    v1 = dict(plants, group3_upright=dict(plants["group3_upright"], depth=360))
    cases = (("group3_upright, depth 400", plants, connected_plants),
             ("group3_upright, v1 depth 360 (comparison only)", v1, connected_plants),
             ("control", plants, reassign(connected_plants, 1, "control")))
    for label, plant_data, chosen in cases:
        specs_c = [opening_spec(p) for p in assign(plant_data, chosen, len(connected.planters))]
        print("dumbbell grows", label)
        try:
            built_c = roof.design_roof(brief, connected, steps=plant_steps_drained(brief, specs_c))
        except (PlanterFitError, ValueError) as error:
            print("    stopped:", type(error).__name__, error)
            continue
        rows_c = drained_planter_rows(brief, connected, specs_c, built_c)
        drains_c = drain_rows(brief, connected, specs_c, built_c)
        print(table(rows_c))
        print(drain_table(drains_c))
        print("all checks pass:", all_ok_drained(rows_c, water_check(built_c.incision), drains_c))
        fig = plan_comparison(brief, connected, roof.design_roof(brief, connected), built_c, specs_c)
        plt.show()
else:
    print("skipped: set RUN_CONNECTED = True in cell 1")